# R09 · Bounded sidecar merge and end-to-end evaluation

This experiment tests whether a fixed-size distribution sidecar can retain its ability to assess an unchanged ARM histogram after repeated merges. It reuses R08's quantized moments and Beta/copula probability calculation. The two compressors use a deterministic spatial-neighbor graph: a geometry-only control and a workload-based reference-distortion heuristic.

**This is an experiment, not an established improvement or accuracy guarantee.** Both compressors, all merge orders, and the controls are reported. Original histogram answers are unchanged. No supervised model is trained.

The end-to-end path estimates H, computes the live assessor score, applies a calibration-only gate, and performs a real inclusive scan of the original raw points on rejection. The exact baseline always performs that same scan. This notebook measures the **Python backend**; it does not claim to have integrated the new sidecar into the team's Java or Spark runtime.

Use the existing real `range_query` data. Small synthetic cases are used only by the automated correctness tests, not as reported performance results.


In [ ]:
# Install only missing dependencies; do not upgrade an existing Databricks stack.
import importlib.util
import subprocess
import sys

_required_packages = ("numpy", "pandas", "scipy", "matplotlib")
_missing_packages = [name for name in _required_packages if importlib.util.find_spec(name) is None]
if _missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing_packages])
print("Required packages are available; LightGBM is not required.")


## Configure paths and the run size

Change the widgets rather than editing source-code paths. No personal workspace path is committed.

- `repo_root`: this Git folder, normally detected automatically.
- `run_dir`: optional completed R01/R08 or related preparation run used to locate histogram exports and split files. Leave it blank when explicit paths are provided or automatic discovery finds the required inputs.
- `histogram_dir`: folder containing `src_0.txt` through `src_3.txt` and `arm_<resolution>.txt`. These are the existing Java histogram exports, not a folder of CSV experiment results.
- `raw_data_dir`: the existing real data folder. `.csv.gz` files are accepted directly.
- `queries_file`: the existing range-query file, with coordinates and offline exact counts. Saved counts are never used as timed exact fallback.
- `split_file`: a saved `outer_splits.csv` / compatible `splits.csv` preserves the previous roles per seed. Without one, the existing query-ID split algorithm is used and the report explicitly labels that choice; it is not grouped validation.

The default pilot matches the verified local configuration: three seeds,32 training probes,128 calibration and256 existing test queries per seed,12 timed queries with two repetitions, and64 additional shared fresh rectangles. It limits query counts while scanning **all source rows** during construction and every exact request. Repeating the fixed fresh seed reproduces the same queries; it is not another independent confirmation. Setting `execution_mode=full` removes the calibration/test quality caps. Timing still uses a deterministic bounded subset. Existing tests have been inspected before, so their results remain exploratory.

`source_partition_factor=1` uses the four original sources. Larger factors partition the same raw rows into more virtual sources for merge-scaling tests; they do not create a new dataset.


In [ ]:
from pathlib import Path
import json


def _detect_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        here = Path("/Workspace") / notebook_path.lstrip("/")
        candidates += [here.parent, *here.parents]
    except Exception:
        pass
    for candidate in candidates:
        if (candidate / "ml" / "sidecar_workflow.py").is_file():
            return str(candidate)
    return str(Path.cwd())


_detected_repo = Path(_detect_repo_root())
_defaults = {
    "repo_root": str(_detected_repo),
    "run_dir": "",
    "histogram_dir": "",
    "raw_data_dir": str(_detected_repo / "range_query"),
    "queries_file": "",
    "split_file": "",
    "output_root": str(_detected_repo.parent / "synopsislake-results"),
    "execution_mode": "pilot",
    "base_resolution": "130",
    "budget_bytes": "54032",
    "seeds": "0,1,2",
    "source_partition_factor": "1",
    "merge_trees": "balanced,left,right",
    "workload_queries": "32",
    "calibration_queries": "128",
    "test_queries": "256",
    "timing_queries": "12",
    "timing_repeats": "2",
    "fresh_query_count": "64",
    "fresh_query_seed": "2026111709",
}
try:
    for key, value in _defaults.items():
        try:
            dbutils.widgets.get(key)
        except Exception:
            dbutils.widgets.text(key, value)
    _widget_values = {key: dbutils.widgets.get(key).strip() for key in _defaults}
except NameError:
    _widget_values = dict(_defaults)
print("Configuration widgets are ready. Review the paths before running the experiment.")


## Validate the configuration

The fixed auxiliary budget includes every serialized mixture record and header. For the workload method, saved development-query geometry and IDs also count against that same budget; the mixture receives only the remaining bytes. The unchanged H is reported separately. Development geometry is needed during merging, not by the query scorer. Temporary merge memory and retained runtime arrays are separate from serialized storage.

Each seed has its own compression workload selected from that seed's training queries only. Calibration queries choose gate thresholds, and test truth does not select compression settings, budgets, or thresholds. Complete score ties are retained. A zero-acceptance violation rate is undefined.


In [ ]:
repo_root = Path(_widget_values["repo_root"]).expanduser().resolve()
if not (repo_root / "ml" / "sidecar_workflow.py").is_file():
    raise FileNotFoundError("Set repo_root to the Git folder containing ml/sidecar_workflow.py")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from ml.sidecar_workflow import run_experiment, package_results

configuration = {key: value for key, value in _widget_values.items()
                 if key != "execution_mode"}
for key in ("base_resolution", "budget_bytes", "source_partition_factor", "workload_queries",
            "calibration_queries", "test_queries", "timing_queries", "timing_repeats",
            "fresh_query_count", "fresh_query_seed"):
    configuration[key] = int(configuration[key])
configuration["seeds"] = [int(value.strip()) for value in configuration["seeds"].split(",") if value.strip()]
configuration["merge_trees"] = [value.strip() for value in configuration["merge_trees"].split(",") if value.strip()]
configuration["modes"] = ["geometry", "workload"]
configuration.update(epsilon=0.25, delta=0.05, min_exact=10)
execution_mode = _widget_values["execution_mode"]
if execution_mode not in ("pilot", "full"):
    raise ValueError("execution_mode must be pilot or full")
if execution_mode == "full":
    configuration["calibration_queries"] = 0
    configuration["test_queries"] = 0
if configuration["timing_queries"] < 1:
    raise ValueError("Keep timing_queries positive; full quality evaluation and timing size are separate")
print(json.dumps(configuration, indent=2))


## Execute on the existing real data

Run this cell once. Each execution creates a unique output directory and preserves failed runs for inspection. Construction and calibration are measured separately. No saved truth table replaces exact execution.

Raw-coordinate loading, moment construction, and fine-grid construction share a parsing pass. Their combined setup cost is recorded in `source_construction.csv` and `run.json`; it is excluded from request latency. An E2E `raw_load` value of zero means the arrays were already loaded by this preparation phase, not that loading was free. Report this setup cost alongside steady-state latency.

`resident` keeps H and assessor artifacts in memory. `per_query` reloads local H/assessor files inside every request timer. The raw arrays are resident for **all methods and both modes**; every exact request still scans every raw row. Neither mode claims a cold OS cache or a Databricks SQL/Spark scan.

If `fresh_query_count > 0`, the workflow freezes the compressed summaries and calibration cutoffs before producing a separate fresh-query evaluation. Fresh rectangles query the same raw dataset; they are not a new dataset. The frozen gates are reused rather than recalibrated on fresh truth.


In [ ]:
result_dir = run_experiment(configuration)
print("Completed:", result_dir)


## Diagnose compression separately from accuracy

The following table compares each compressed reference with the uncompressed input reference on the saved test geometries. It uses no truth to choose a compressor and changes no gate. A small distortion would only mean that compression preserves the reference; it would not prove that either reference is correct. Virtual-partition runs use their matching uncompressed virtual union for this comparison.

Keep the per-seed rows: repeated queries across seeds are not independent observations.


In [ ]:
import numpy as np
import pandas as pd

_saved_quality = pd.read_csv(Path(result_dir) / "per_query_quality.csv", float_precision="round_trip")
_test_quality = _saved_quality[_saved_quality["split"] == "test"].copy()
_reference_method = ("uncompressed_virtual_partition_reference"
                     if "uncompressed_virtual_partition_reference" in set(_test_quality.method)
                     else "uncompressed_R08_reference")
_keys = ["family", "seed", "qId"]
_reference_rows = _test_quality[_test_quality.method == _reference_method][
    _keys + ["reference_count", "score"]].rename(columns={
        "reference_count": "uncompressed_reference_count", "score": "uncompressed_score"})
if _reference_rows.duplicated(_keys).any():
    raise ValueError("Duplicate uncompressed reference geometry/seed rows")
_distortion_rows = []
for _method, _part in _test_quality[_test_quality.method.str.startswith("bounded_")].groupby("method"):
    _joined = _part.merge(_reference_rows, on=_keys, validate="many_to_one")
    if len(_joined) != len(_part):
        raise ValueError("Compression diagnostic lost reference rows")
    for (_family, _seed), _group in _joined.groupby(["family", "seed"]):
        _count_gap = np.abs(_group.reference_count.to_numpy() - _group.uncompressed_reference_count.to_numpy())
        _score_gap = np.abs(_group.score.to_numpy() - _group.uncompressed_score.to_numpy())
        _distortion_rows.append(dict(
            family=_family, seed=int(_seed), method=_method, reference_method=_reference_method,
            unique_queries=int(_group.qId.nunique()),
            mean_absolute_reference_count_distortion=float(_count_gap.mean()),
            mean_relative_reference_distortion=float(np.mean(_count_gap / np.maximum(_group.uncompressed_reference_count.to_numpy(), 1.0))),
            mean_proxy_score_distortion_pp=float(100 * _score_gap.mean()),
            p90_proxy_score_distortion_pp=float(100 * np.quantile(_score_gap, 0.90)),
        ))
compression_distortion = pd.DataFrame(_distortion_rows)
compression_distortion.to_csv(Path(result_dir) / "compression_distortion.csv", index=False)
display(compression_distortion) if "display" in globals() else print(compression_distortion.to_string(index=False))


## Inspect quality, merge costs, and end-to-end latency together

Read the analysis and tables together. A low selected-answer violation rate with very few accepts is not sufficient evidence of useful speedup. A high AUC does not imply precise numerical error predictions.

Check that all merge nodes preserve mass and respect their byte cap. Different merge trees can produce different summaries. The workload compressor is a local heuristic, not an optimality or accuracy guarantee.


In [ ]:
import pandas as pd

for _table_name in ("summary.csv", "metrics.csv", "e2e_summary.csv", "merge_audit.csv", "payload_costs.csv"):
    _table_path = Path(result_dir) / _table_name
    if _table_path.is_file():
        print(_table_name)
        _table = pd.read_csv(_table_path)
        display(_table) if "display" in globals() else print(_table.to_string(index=False))
print("English analysis:", Path(result_dir) / "analysis_en.md")
print("All output files:")
for _path in sorted(Path(result_dir).iterdir()):
    print(_path.name)


## Export for review

Share the result ZIP. It contains the frozen protocol, split and workload identities, quality/gate counts, merge/storage audits, per-query timings, summary tables, and English analysis. The raw point files are not included.

This notebook does not deploy anything, create a Databricks job, modify the Java histogram core, or retrain earlier models.


In [ ]:
archive_path = package_results(result_dir)
print("Share this file:", archive_path)
print("Retained full run directory:", result_dir)
